# Đánh giá mô hình hồi quy: MAE, MSE, RMSE, $R^2$

Tiếp tục ví dụ dự đoán cân nặng theo chiều cao: huấn luyện trên 13 người, đánh giá trên 2 người cao **155 cm** và **160 cm**.

| Độ đo | Công thức | Ý nghĩa |
| --- | --- | --- |
| MAE | $\frac{1}{N}\sum \lvert y_i - \hat{y}_i \rvert$ | Sai số tuyệt đối trung bình |
| MSE | $\frac{1}{N}\sum (y_i - \hat{y}_i)^2$ | Sai số bình phương trung bình |
| RMSE | $\sqrt{\text{MSE}}$ | Cùng đơn vị với $y$, dễ diễn giải |
| $R^2$ | $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ | Tỉ lệ phương sai được giải thích (càng gần 1 càng tốt) |

## 1. Huấn luyện mô hình

In [1]:
import numpy as np
from sklearn import linear_model

# Tập huấn luyện: 13 người, mỗi hàng là một mẫu
X = np.array([[147, 150, 153, 158, 163, 165, 168, 170, 173, 175, 178, 180, 183]]).T   # chiều cao (cm)
y = np.array([49, 50, 51, 54, 58, 59, 60, 62, 63, 64, 66, 67, 68])                     # cân nặng (kg)

model = linear_model.LinearRegression()
model.fit(X, y)

print("w1 =", model.coef_[0])
print("w0 =", model.intercept_)

w1 = 0.5592049619396674
w0 = -33.73541020580774


## 2. Đánh giá trên tập kiểm tra bằng scikit-learn

In [2]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_test = np.array([[155], [160]])   # chiều cao của tập kiểm tra
y_test = np.array([52, 56])         # cân nặng thực tế
y_pred = model.predict(X_test)      # cân nặng dự đoán

print("MAE  =", mean_absolute_error(y_test, y_pred))
print("MSE  =", mean_squared_error(y_test, y_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred)))

MAE  = 0.6019875951508311
MSE  = 0.4775619437686744
RMSE = 0.6910585675387249


## 3. Tính tay theo công thức để hiểu rõ

Sai số của từng mẫu: $e_i = y_i - \hat{y}_i$.

In [3]:
errors = y_test - y_pred   # sai số e_i của từng mẫu
for height, actual, predicted, error in zip(X_test.ravel(), y_test, y_pred, errors):
    print(f"Chiều cao {height} cm: thực tế {actual} kg, dự đoán {predicted:.4f} kg, sai số e = {error:+.4f}")

mae = np.mean(np.abs(errors))   # trung bình trị tuyệt đối sai số
mse = np.mean(errors ** 2)      # trung bình bình phương sai số
rmse = np.sqrt(mse)             # căn bậc hai của MSE, cùng đơn vị kg

print("\nMAE  =", mae)
print("MSE  =", mse)
print("RMSE =", rmse)

Chiều cao 155 cm: thực tế 52 kg, dự đoán 52.9414 kg, sai số e = -0.9414
Chiều cao 160 cm: thực tế 56 kg, dự đoán 55.7374 kg, sai số e = +0.2626

MAE  = 0.6019875951508311
MSE  = 0.4775619437686744
RMSE = 0.6910585675387249


**Nhận xét:**

- MAE ≈ 0.6 kg: trung bình mô hình dự đoán lệch khoảng 0.6 kg.
- RMSE ≥ MAE luôn đúng; RMSE phạt nặng hơn các sai số lớn vì bình phương trước khi lấy trung bình.
- MSE có đơn vị kg², khó diễn giải, nên thường báo cáo RMSE.

## 4. Hệ số xác định $R^2$

Tập kiểm tra chỉ có 2 mẫu nên $R^2$ trên đó không đáng tin; ở đây tính $R^2$ trên **tập huấn luyện** để minh họa.

In [4]:
y_train_pred = model.predict(X)   # dự đoán trên tập huấn luyện

r2 = r2_score(y, y_train_pred)
r2_manual = 1 - np.sum((y - y_train_pred) ** 2) / np.sum((y - y.mean()) ** 2)   # tính theo công thức

print("R^2 (scikit-learn) =", r2)
print("R^2 (tính tay)     =", r2_manual)

R^2 (scikit-learn) = 0.9944856921763271
R^2 (tính tay)     = 0.9944856921763271


$R^2$ rất gần 1: đường thẳng giải thích gần như toàn bộ sự biến thiên của cân nặng trong tập huấn luyện.